# Investigating Inference and Training Strategies for Terminology Translation

This notebook contains the complete pipeline using Google Colab for our WMT25 Terminology Translation project. We investigate how to force a lightweight model (Qwen2.5-1.5B-Instruct) to adhere to terminology dictionaries using four distinct research questions (RQs).

**Pipeline Overview:**
1. Environment Setup & Model Loading
2. RQ1: Inference-Time Strategies (Conditional Prompting)
3. RQ2: Training Data Curation (Synthesizing data from News Commentary)
4. RQ3: Model Adaptation (LoRA Fine-Tuning)
5. RQ4: Multi-Step Refinement (Dual-stage reasoning)
6. Post-processing & Output Generation

In [ ]:
!git clone https://github.com/Sethjsa/nlp2-26.git

Cloning into 'nlp2-26'...
remote: Enumerating objects: 2104, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 2104 (delta 6), reused 4 (delta 4), pack-reused 2096 (from 2)
Receiving objects: 100% (2104/2104), 110.61 MiB | 14.63 MiB/s, done.
Resolving deltas: 100% (1426/1426), done.
Updating files: 100% (1976/1976), done.


In [ ]:
%cd nlp2-26

/content/nlp2-26


In [ ]:
!pip install -r requirements.txt

In [ ]:
!pip install -q -U transformers datasets trl peft accelerate bitsandbytes sacrebleu jsonlines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.2 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import drive

# Check whether Drive is already mounted; if not, mount it
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully!")
else:
    print("✅ Drive is already mounted. Skipping this step.")

✅ Drive is already mounted. Skipping this step.


## RQ1: Inference-Time Strategies (Zero-Shot Prompting)
In this section, we evaluate the baseline capability of the `Qwen2.5-1.5B-Instruct` base model. We implement a conditional prompt strategy. If terminology is provided, the model receives a strict system prompt acting as a professional technical translator to use the provided terms directly. If no terminology is provided, it defaults to a standard translation prompt.

In [ ]:
import json
import os
import torch
import random
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model

# Load base model and tokenizer
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:85: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading base model...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
from tqdm import tqdm
import torch

# Fix the test split logic in starter.ipynb by loading the full dataset
def load_full_test_data(mode, language_pair="ende"):
    test_jsonl_path = f"test-data/track1/{language_pair}.{mode}.jsonl"
    data = []
    with open(test_jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            entry = json.loads(line.strip("\n"))
            terms = ""
            if mode == "proper":
                terms = entry.get("terms", {})
            elif mode == "random":
                terms = entry.get("terms", {})

            # Convert dictionary terms into plain text for prompting
            term_str = ", ".join([f"'{k}' -> '{v}'" for k, v in terms.items()]) if terms else ""
            data.append({"en": entry["en"], "terms": term_str})
    return data

# Save translations in the format required by the WMT25 terminology evaluation script
def save_system_predictions(translations, mode, system_name, language_pair="ende"):
    output_dir = f"wmt25-terminology/ranking/submissions/track1/{system_name}"
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{system_name}.{language_pair}.{mode}.jsonl"

    test_data = load_full_test_data(mode, language_pair)
    with open(output_path, "w", encoding="utf-8") as f:
        for original, pred in zip(test_data, translations):
            f.write(json.dumps({
                "en": original["en"],
                "terms": original["terms"],
                "de": pred.replace("\n", " ").strip()
            }) + "\n")
    print(f"[{system_name}] Saved {len(translations)} predictions to {output_path}")

def run_batch_inference(model, data, system_name, mode):
    model.eval()
    translations = []
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    for entry in tqdm(data, desc=f"Inferencing {system_name} - {mode}"):
        source = entry["en"]
        terms = entry["terms"]

        if terms:
            system_msg = "You are a professional translator. You MUST strictly substitute the specified terminology in your translation."
            prompt = f"Translate the following text to German.\nSource: {source}\nMandatory Terminology: {terms}\nTranslation:"
        else:
            system_msg = "You are a professional translator."
            prompt = f"Translate the following text to German.\nSource: {source}\nTranslation:"

        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt}
        ]
        text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text_input, return_tensors="pt").to(model.device)

        with torch.inference_mode():
            outputs = model.generate(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )


        pred = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        translations.append(pred)

    save_system_predictions(translations, mode, system_name)



In [ ]:
print("=== Running RQ1: Base Model Inference ===")
for mode in ["noterm", "proper", "random"]:
    print(f"Processing mode: {mode}")
    test_data = load_full_test_data(mode)
    run_batch_inference(base_model, test_data, "RQ1_Base", mode)

=== Running RQ1: Base Model Inference ===
Processing mode: noterm


Inferencing RQ1_Base - noterm: 100%|██████████| 500/500 [19:20<00:00,  2.32s/it]


[RQ1_Base] Saved 500 predictions to wmt25-terminology/ranking/submissions/track1/RQ1_Base/RQ1_Base.ende.noterm.jsonl
Processing mode: proper


Inferencing RQ1_Base - proper: 100%|██████████| 500/500 [18:38<00:00,  2.24s/it]


[RQ1_Base] Saved 500 predictions to wmt25-terminology/ranking/submissions/track1/RQ1_Base/RQ1_Base.ende.proper.jsonl
Processing mode: random


Inferencing RQ1_Base - random: 100%|██████████| 500/500 [18:40<00:00,  2.24s/it]

[RQ1_Base] Saved 500 predictions to wmt25-terminology/ranking/submissions/track1/RQ1_Base/RQ1_Base.ende.random.jsonl


## RQ2: Training Data Curation (Data Synthesis)
To prepare the model for fine-tuning, we synthesize a dataset that links prompt constraints to specific target outputs. We stream the WMT News Commentary dataset and randomly inject authentic terminology pairs extracted from the project development set. We generate exactly 5000 structured examples to provide sufficient learning signals while maintaining computational efficiency.

## RQ3: Model Adaptation (LoRA Fine-Tuning)
We apply Parameter-Efficient Fine-Tuning (PEFT) to the base model using the synthetic dataset curated in RQ2. We utilize Low-Rank Adaptation (LoRA) targeting the attention projection modules (rank 16, alpha 32). The model is trained for a single epoch to prevent overfitting.

In [ ]:
import urllib.request
import gzip
import random
import json
from datasets import Dataset
import os
from transformers.trainer_utils import get_last_checkpoint
import json


print("=== Running RQ2: Data Synthesis ===")

dev_terms = {}

with open("dev-data/ende_dev.jsonl", 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)

        if 'proper_terms' in data:
            dev_terms.update(data['proper_terms'])

term_items = list(dev_terms.items())


print("Downloading and parsing news-commentary directly...")
hf_path = "wmt/news-commentary"
data_file = "v14/training/news-commentary-v14.de-en.tsv.gz"
url = f"https://huggingface.co/datasets/{hf_path}/resolve/main/{data_file}"

sft_data = []

# Stream and parse the file directly
with urllib.request.urlopen(url) as response:
    with gzip.GzipFile(fileobj=response) as f:
        for line in f:
            # Decode and strictly split by tab
            decoded_line = line.decode('utf-8').strip('\n')
            parts = decoded_line.split('\t')

            # Skip any bad lines automatically
            if len(parts) == 2:
                de_text, en_text = parts

                # --- Synthesis Logic ---
                if random.random() < 0.3 and term_items:
                    sampled_terms = dict(random.sample(term_items, k=random.randint(1, 2)))
                    dict_str = ", ".join([f"'{k}' -> '{v}'" for k, v in sampled_terms.items()])
                    prompt = f"Translate the following English sentence to German, strictly respecting the given terminology.\nSource: {en_text}\nTerminology: {dict_str}\nTranslation:"
                else:
                    prompt = f"Translate the following English sentence to German.\nSource: {en_text}\nTranslation:"

                messages = [{"role": "user", "content": prompt}, {"role": "assistant", "content": de_text}]

                # 'tokenizer' is defined earlier
                text = tokenizer.apply_chat_template(messages, tokenize=False)
                sft_data.append({"text": text})

            # Stop once we have exactly 5000 valid samples
            if len(sft_data) >= 5000:
                break

train_dataset = Dataset.from_list(sft_data)
print(f"Successfully generated {len(train_dataset)} training examples.")

print("=== Running RQ3: LoRA Fine-Tuning ===")
if hasattr(base_model, "unload"):
    try:
        base_model = base_model.unload()
    except Exception:
        pass

peft_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
model_sft = get_peft_model(base_model, peft_config)

# Configure training output directory on Google Drive
drive_output_dir = "/content/drive/MyDrive/LLM_Training/terminology-sft"

training_config = SFTConfig(
    output_dir=drive_output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    dataset_text_field="text",
    save_steps=10,                      # Save every 10 steps
    report_to="none"
)

trainer = SFTTrainer(
    model=model_sft,
    train_dataset=train_dataset,
    args=training_config,
    processing_class=tokenizer
)

# Check whether a saved checkpoint already exists in Drive
last_checkpoint = None
if os.path.isdir(drive_output_dir):
    last_checkpoint = get_last_checkpoint(drive_output_dir)

if last_checkpoint is not None:
    print(f"🔄 Checkpoint detected. Resuming training from {last_checkpoint}...")
else:
    print("▶️ No checkpoint detected. Starting fresh training...")

trainer.train(resume_from_checkpoint=last_checkpoint)

# After training, save the final LoRA weights and tokenizer to Drive
final_save_path = "/content/drive/MyDrive/LLM_Training/terminology-sft-final"
trainer.model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)

print(f"✅ Model safely saved to: {final_save_path}")



=== Running RQ2: Data Synthesis ===
Successfully generated 5000 training examples.
=== Running RQ3: LoRA Fine-Tuning ===


Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🔄 Checkpoint detected. Resuming training from /content/drive/MyDrive/LLM_Training/terminology-sft/checkpoint-157...


[transformers] Warning: The following arguments do not match the ones in the `trainer_state.json` within the checkpoint directory: 
	save_steps: 10 (from args) != 1000 (from trainer_state.json)


Step,Training Loss


✅ Model safely saved to: /content/drive/MyDrive/LLM_Training/terminology-sft-final


In [ ]:
# Fine-tuned model inference
print("=== Running RQ3 Inference (Auto-Resume) ===")

output_dir = "wmt25-terminology/ranking/submissions/track1/RQ3_SFT"
expected_count = 500

for mode in ["noterm", "proper", "random"]:
    output_file = os.path.join(output_dir, f"RQ3_SFT.ende.{mode}.jsonl")

    # Check whether the file exists and whether the line count reaches the expected amount
    if os.path.exists(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            line_count = sum(1 for _ in f)

        if line_count >= expected_count:
            print(f"✅ Skipping {mode}: Complete results detected ({line_count} entries) -> {output_file}")
            continue
        else:
            print(f"⚠️ {mode} file is incomplete ({line_count}/{expected_count}). Re-running this mode...")
            # Remove the incomplete file before rerunning to avoid append-write issues
            os.remove(output_file)

    print(f"▶️ Starting inference mode: {mode}")
    test_data = load_full_test_data(mode)
    run_batch_inference(trainer.model, test_data, "RQ3_SFT", mode)

## RQ4: Multi-Step Refinement
This section implements our advanced dual-step reasoning approach. We decouple the translation task into two phases. Step 1 focuses purely on fluency without dictionary constraints. Step 2 feeds the draft back to the model with strict instructions to act as a post-editor and insert the provided terminology. We run this approach on both the base model and the fine-tuned model (temporarily disabling the LoRA adapter for the base evaluation).

In [ ]:
print("=== Running RQ4: Multi-Step Refinement ===")

def run_multistep_inference(model, data, system_name, mode):
    translations = []
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    for entry in tqdm(data, desc=f"Inferencing {system_name} - {mode}"):

        source = entry["en"]
        terms = entry["terms"]

        # Step 1: Translation only
        step1_prompt = f"Translate the following English sentence to German.\nSource: {source}\nTranslation:"
        msg1 = [{"role": "user", "content": step1_prompt}]
        in1 = tokenizer(tokenizer.apply_chat_template(msg1, tokenize=False, add_generation_prompt=True), return_tensors="pt").to(model.device)

        with torch.inference_mode():
            out1 = model.generate(input_ids=in1.input_ids, attention_mask=in1.attention_mask, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id, use_cache=True,)
        draft = tokenizer.decode(out1[0][in1.input_ids.shape[1]:], skip_special_tokens=True).strip()

        final = draft
        if terms:
            # Step 2: Terminology review and correction
            # Only executed when terminology constraints exist
            step2_prompt = f"Draft translation: '{draft}'\nCritically revise this draft so it strictly incorporates these exact terms: {terms}. Output only the final German sentence."
            msg2 = [
                {"role": "system", "content": "You are a precise post-editor. Follow dictionary constraints flawlessly."},
                {"role": "user", "content": step2_prompt}
            ]

            in2 = tokenizer(tokenizer.apply_chat_template(msg2, tokenize=False, add_generation_prompt=True), return_tensors="pt").to(model.device)

            with torch.inference_mode():
                out2 = model.generate(input_ids=in2.input_ids, attention_mask=in2.attention_mask, max_new_tokens=128, do_sample=False, pad_token_id=tokenizer.eos_token_id, use_cache=True,)
            final = tokenizer.decode(out2[0][in2.input_ids.shape[1]:], skip_special_tokens=True).strip()

        translations.append(final)

    save_system_predictions(translations, mode, system_name)

modes = ["noterm", "proper", "random"]

for mode in modes:

  # if mode == "random":

    test_data = load_full_test_data(mode)

    # 1. Test the untuned base model
    #    (to evaluate the effectiveness of the multi-step prompting itself)
    with trainer.model.disable_adapter():
        # run_batch_inference(base_model, test_data, "RQ1_Base", mode)
        run_multistep_inference(trainer.model, test_data, "RQ4_MultiStep_Base", mode)


    # 2. Test the fine-tuned model
    #    (to evaluate the performance of the integrated approach)
    # run_batch_inference(trainer.model, test_data, "RQ3_SFT", mode)
    run_multistep_inference(trainer.model, test_data, "RQ4_MultiStep_SFT", mode)


=== Running RQ4: Multi-Step Refinement ===


Inferencing RQ4_MultiStep_Base - random: 100%|██████████| 500/500 [35:19<00:00,  4.24s/it]


[RQ4_MultiStep_Base] Saved 500 predictions to wmt25-terminology/ranking/submissions/track1/RQ4_MultiStep_Base/RQ4_MultiStep_Base.ende.random.jsonl


Inferencing RQ3_SFT - random: 100%|██████████| 500/500 [15:54<00:00,  1.91s/it]


[RQ3_SFT] Saved 500 predictions to wmt25-terminology/ranking/submissions/track1/RQ3_SFT/RQ3_SFT.ende.random.jsonl


Inferencing RQ4_MultiStep_SFT - random: 100%|██████████| 500/500 [32:59<00:00,  3.96s/it]

[RQ4_MultiStep_SFT] Saved 500 predictions to wmt25-terminology/ranking/submissions/track1/RQ4_MultiStep_SFT/RQ4_MultiStep_SFT.ende.random.jsonl


In [ ]:
!zip -r track1.zip /content/nlp2-26/wmt25-terminology/ranking/submissions/track1/
!cp track1.zip /content/drive/MyDrive/

  adding: content/nlp2-26/wmt25-terminology/ranking/submissions/track1/ (stored 0%)
  adding: content/nlp2-26/wmt25-terminology/ranking/submissions/track1/RQ3_SFT/ (stored 0%)
  adding: content/nlp2-26/wmt25-terminology/ranking/submissions/track1/RQ3_SFT/RQ3_SFT.ende.random.jsonl (deflated 70%)
  adding: content/nlp2-26/wmt25-terminology/ranking/submissions/track1/TEAMNAME/ (stored 0%)
  adding: content/nlp2-26/wmt25-terminology/ranking/submissions/track1/TEAMNAME/TEAMNAME.ende.proper.jsonl (deflated 65%)
  adding: content/nlp2-26/wmt25-terminology/ranking/submissions/track1/.DS_Store (deflated 97%)
  adding: content/nlp2-26/wmt25-terminology/ranking/submissions/track1/RQ4_MultiStep_SFT/ (stored 0%)
  adding: content/nlp2-26/wmt25-terminology/ranking/submissions/track1/RQ4_MultiStep_SFT/RQ4_MultiStep_SFT.ende.random.jsonl (deflated 69%)
  adding: content/nlp2-26/wmt25-terminology/ranking/submissions/track1/RQ4_MultiStep_Base/ (stored 0%)
  adding: content/nlp2-26/wmt25-terminology/rank

## Post-Processing: Output Cleaning
We found that the model could exhibit a conversational noise, encapsulating translations within redundant explanatory text. This script applies a heuristic cleaning process (using regular expressions and source text regurgitation filtering) to isolate the raw German output prior to scoring evaluation.

In [ ]:
import json
import os
import re

def clean_noise_explanation(text, source_en=""):
    if not text:
        return ""

    text = text.strip()
    original_text = text

    # 0. Remove source English text regurgitation
    if source_en:
        src_clean = source_en.strip()
        escaped_src = re.escape(src_clean)
        # Match optional punctuation/spaces at the beginning,
        # followed by the repeated source text and optional trailing punctuation/spaces
        src_pattern = r'^[\s"\'„“\-\*]*' + escaped_src + r'[\s"\'”\-\*]*'
        # Ignore case and completely strip the repeated source text
        text = re.sub(src_pattern, '', text, flags=re.IGNORECASE).strip()

    # 1. Precise prefix stripping
    # Use regex anchored at the beginning while allowing periods in the original sentence
    intro_patterns = [
        # Match patterns like "Sure! Here is the translation:" or "Here's the output:"
        r'^[\s*]*(?:sure|certainly|yes|of course|ok|here(?: is|\'s)?\b)[^:\n]*[:\n]\s*',
        # Match patterns like "The source translation of ... into German is:"
        r'^[\s*]*(?:the\s+)?(?:source\s+|original\s+)?(?:translation|german|english|output)[^:\n]*[:\n]\s*'
    ]
    for p in intro_patterns:
        text = re.sub(p, '', text, flags=re.IGNORECASE).strip()

    # 2. Remove trailing parenthetical notes
    # Must be processed early to avoid internal colons interfering with later logic
    # Match ending (...) or [...] containing keywords like ai, translation, generated, etc.
    text = re.sub(r'\s*[\(\[][^\)\]]*(?:translation|\bai\b|generated|provided|english)[^\)\]]*[\)\]]\s*$', '', text, flags=re.IGNORECASE)

    # 3. Extended suffix truncation
    trash_markers = [
        "This translation", "The target", "Note:", "Note that",
        "In German", "This directly", "The translation",
        "This implies", "Here,", "This means", "As requested",
        "This translates", "This maintains", "It translates",
        "This word", "The term", "This phrase", "In this context",
        "This version", "The German", "The English", "This accurately",
        "This conveys", "This indicates", "The source",
        "The original", "translated", "this is"
    ]
    for marker in trash_markers:
        match = re.search(re.escape(marker), text, re.IGNORECASE)
        if match:
            text = text[:match.start()].strip()

    # 4. Extract content wrapped in leading quotation marks and filter residual explanatory suffixes
    if text.startswith('"') or text.startswith('„') or text.startswith("'"):
        start_quote = text[0]
        end_quote = '"' if start_quote == '"' else ('“' if start_quote == '„' else "'")

        end_idx = text.rfind(end_quote)
        if end_idx > 0:
            if end_idx == len(text) - 1:
                # Fully wrapped by quotes -> simply remove outer quotes
                text = text[1:-1].strip()
            else:
                # Additional trailing content exists after the quote
                # Detect whether it is explanatory text
                trailing_part = text[end_idx+1:].lower()
                trailing_garbage = r'\b(indicating|meaning|translates|conveys|pertains|refers|allows|used for|because|therefore)\b'
                # If trailing content looks like explanation,
                # or contains no meaningful alphabetic characters,
                # safely extract only the quoted part
                if re.search(trailing_garbage, trailing_part) or not re.search(r'[a-zäöüß]', trailing_part):
                    text = text[1:end_idx].strip()

    # 5. Remove residual Markdown bold markers
    text = text.replace("**", "").strip()

    # 6. Final fallback cleanup for surrounding single/double quotes
    if (text.startswith('"') and text.endswith('"')) or (text.startswith("'") and text.endswith("'")):
        text = text[1:-1].strip()

    # Safety fallback
    if len(re.sub(r'[\W_]', '', text)) < 3:
        return original_text.strip()

    return text

def patch_and_clean(system_name, mode, language_pair="ende"):
    """
    Restore the terminology dictionary from the original test set
    and clean noisy German translations.
    """
    original_test_path = f"test-data/track1/{language_pair}.{mode}.jsonl"
    noisy_pred_path = f"wmt25-terminology/ranking/submissions/track1/{system_name}/{system_name}.{language_pair}.{mode}.jsonl"
    fixed_output_path = f"wmt25-terminology/ranking/submissions/track1/{system_name}/{system_name}.{language_pair}.{mode}.fixed.jsonl"

    if not os.path.exists(noisy_pred_path):
        print(f"[Warning] File not found: {noisy_pred_path}. Skipping this mode.")
        return

    # 2. Read the original test set and recover terminology dictionaries
    raw_terms_list = []

    with open(original_test_path, "r", encoding="utf-8") as f:
        for line in f:
            entry = json.loads(line)

            if mode == "noterm":
                raw_terms_list.append("") # Keep unchanged for noterm mode
            else:
                # Official data uses unified field name: "terms"
                raw_terms_list.append(entry.get("terms", {}))

    # 3. Read noisy translations, clean German text, and restore terminology dictionaries
    fixed_entries = []
    with open(noisy_pred_path, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            pred_entry = json.loads(line)

            # Clean German translation output
            raw_de = pred_entry.get("de", "")
            cleaned_de = clean_noise_explanation(raw_de, pred_entry["en"])

            # Construct output format aligned with "tower" format
            out_dict = {
                "en": pred_entry["en"],
                "de": cleaned_de
            }
            # Restore terminology dictionary
            if mode != "noterm" and idx < len(raw_terms_list):
                out_dict["terms"] = raw_terms_list[idx]

            fixed_entries.append(out_dict)

    # Overwrite the original prediction file
    with open(noisy_pred_path, "w", encoding="utf-8") as f:
        for entry in fixed_entries:
            f.write(json.dumps(entry) + "\n")

    print(f"[{system_name} - {mode}] Successfully restored terminology dictionaries and cleaned {len(fixed_entries)} entries.")


# Run the data repair pipeline
my_systems = ["RQ1_Base", "RQ3_SFT", "RQ4_MultiStep_Base", "RQ4_MultiStep_SFT"]
test_modes = ["noterm", "proper", "random"]

for system in my_systems:
    for mode in test_modes:
        patch_and_clean(system, mode)

print("\nAll prediction files have been successfully repaired! You can now directly run the official evaluation script.")

[RQ1_Base - noterm] Successfully restored terminology dictionaries and cleaned 500 entries.
[RQ1_Base - proper] Successfully restored terminology dictionaries and cleaned 500 entries.
[RQ1_Base - random] Successfully restored terminology dictionaries and cleaned 500 entries.
[RQ3_SFT - noterm] Successfully restored terminology dictionaries and cleaned 500 entries.
[RQ3_SFT - proper] Successfully restored terminology dictionaries and cleaned 500 entries.
[RQ3_SFT - random] Successfully restored terminology dictionaries and cleaned 500 entries.
[RQ4_MultiStep_Base - noterm] Successfully restored terminology dictionaries and cleaned 500 entries.
[RQ4_MultiStep_Base - proper] Successfully restored terminology dictionaries and cleaned 500 entries.
[RQ4_MultiStep_Base - random] Successfully restored terminology dictionaries and cleaned 500 entries.
[RQ4_MultiStep_SFT - noterm] Successfully restored terminology dictionaries and cleaned 500 entries.
[RQ4_MultiStep_SFT - proper] Successfully re

In [ ]:
!find /content/nlp2-26/ -type d -name ".ipynb_checkpoints" -exec rm -rf {} +

## Evaluation
We used the official evaluation scripts provided by the WMT25 Terminology Shared Task repository. Note: The evaluation scripts contains several issues that need to be fixed.

We only evaluate English-to-German (ende), so in evaluate_qual_acc_track1.py, on line 4, the loop for lang in ["de", "es", "ru"] should be updated to keep only "de". On line 84, stats_dict, error_files = run_cycle(lang, "en", mode, f'../submissions/track1') should swap lang and "en".

In termbasedmetric.py, line 334, proper_inside is undefined. To fix this, add proper_inside=False at the beginning of the _retrieve_predefined_terms function (line 315).

In [ ]:
%cd /content/nlp2-26/wmt25-terminology/ranking/metric_track1/
# 1. Evaluate general translation quality (ChrF) and terminology accuracy (Acc)
!python evaluate_qual_acc_track1.py


Streaming output truncated to the last 5000 lines.
        English term: data provider
        German translation: Als Datenprovider legen Sie das Preismodell für ein Datenprodukt fest: Ein Datenprodukt kann entweder kostenlos sein oder den Kauf einer Lizenz zu einem bestimmten Preis erfordern.
        Translated term: Datenprovider

English sentence: The data product is still available in the provider's data product list.
        English term: provider
        German translation: Das Datenprodukt ist weiterhin in der Datenproduktliste des Providers verfügbar.
        Translated term: Providers

 

 English sentence: In this section, you can find information about the security features of , freight collaboration option. 
 English term: collaboration 
 Translation: In diesem Abschnitt finden Sie Informationen über die Sicherheitsmerkmale der Kombinatorie und die Kooperationsoption. 
 Translated term: 1023

English sentence: In the case of an emergency, the fire department will call the 

In [ ]:
!cp /content/nlp2-26/wmt25-terminology/ranking/metric_track1/scores/track1_score_dict.json /content/drive/MyDrive/

updating: content/nlp2-26/wmt25-terminology/ranking/metric_track1/scores/track1_score_dict.json (deflated 80%)


We also need to run three modes consecutively. The original consistency_script_track1.py overwrites previous outputs, so the script must be updated to avoid overwriting saved results.

In [ ]:
%%writefile consistency_script_track1.py
import pandas as pd
import os
import json
import time

import nltk
nltk.download('punkt_tab')

from termbasedmetric import TermBasedMetric

import argparse

def run_cycle(src_lang, tgt_lang, mode, filepath):
    '''
    runs evaluation for the 1st track, for a given source, target language and mode,
        collecting the statistics and the extracted data (terms, alignments, pseudoreferences)

    :param src_lang: source language
    :param tgt_lang: target language
    :param mode: evaluation mode ('proper', 'random', 'noterm')
    :param filepath: path to the directory containing the submitted files
    '''

    # collect all statistics in the format: {system: {pseudoref_choice: {macro: score, micro: score}}}
    total_stats = {}
    # read all submitted files
    print(os.listdir(filepath))

    file_list = []
    for dir1 in os.listdir(filepath):
        dir1_path = os.path.join(filepath, dir1)
        if os.path.isdir(dir1_path):
            for f in os.listdir(dir1_path):
                # Only include files matching the current source language, target language, and mode
                if f.endswith('.jsonl') and (src_lang + tgt_lang) in f and mode in f:
                    file_list.append(os.path.join(dir1, f))

    print("file_list: ", file_list)
    # file_list = [f for f in os.listdir(filepath) if f.endswith('.jsonl') and src_lang+tgt_lang in f and mode in f]

    # initializing term based metric
    TBM = TermBasedMetric(src_lang, tgt_lang, 'predefined', 'llm')

    # catching files with raised errors (deprecated)
    error_files = []
    print("file_list: ", file_list)
    for file in file_list:
        print(file)

        system_name = file.split('.')[0] #.split('/')[-1]
        # loading the submitted file
        TBM.load(system_name, mode=mode, file_path='../submissions/')
        # extracting keywords
        TBM.extract_keywords()
        # aligning source terms to translated sentences
        print("aligning with qwen2.5-0.5b-instruct: ", system_name)
        TBM.align(test=True)
        pseudoref_choices = ['first', 'frequent', 'predefined']
        var_dict = {pseudoref_choice: {'micro': 0, 'macro': 0} for pseudoref_choice in pseudoref_choices}
        flat_df = []
        for pseudoref_choice in pseudoref_choices:
            # assigning pseudoreferences for all three pseudoreference types (first, frequent, predefined)
            _, _, _, sel = TBM.assign_pseudoreferences(pseudoref_choice)
            # computing the micro and macro averaged accuracies for pseudoreference and aligned term overlaps
            micro, flat_micro = TBM.compute_metric('micro')
            macro, _ = TBM.compute_metric('macro')
            if pseudoref_choice == 'first':
                sel_first = flat_micro.copy(deep=True)
            elif pseudoref_choice == 'frequent':
                sel_freq = flat_micro.copy(deep=True)
            flat_micro.rename(columns={'pseudoref': f'pseudoref_{pseudoref_choice}'}, inplace=True)
            flat_df.append(flat_micro)
            var_dict[pseudoref_choice]['micro'] = micro
            var_dict[pseudoref_choice]['macro'] = macro
        # adding all statistics to total dict
        total_stats[system_name] = var_dict
        flat_df = pd.concat(flat_df, axis=1)
        # saving the file with pseudoreferences (and aligned term translations)
        path = f'pseudorefs/{system_name}_pseudoref.tsv'
        os.makedirs(os.path.dirname(path), exist_ok=True)
        flat_df.to_csv(path, sep='\t')
        # saving the file with all initial data (sentences, terms, extracted translations)
        path = f'processed/{system_name}_processed.tsv'
        os.makedirs(os.path.dirname(path), exist_ok=True)
        TBM.bitext_df[['terms', 'src_raw', 'mt_raw', 'src_terms', 'alg_terms', 'norm_tgt_terms']].to_csv(path, sep='\t')

    return total_stats, error_files


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument('-s', '--srclang')
    parser.add_argument('-t', '--tgtlang')
    parser.add_argument('-m', '--mode')
    # parser.add_argument('-l', '--track')
    args=parser.parse_args()
    print(args.srclang, args.tgtlang, args.mode)
    a = time.time()
    # stats_dict, error_files = run_cycle(args.srclang, args.tgtlang, args.mode, f'data/submissions/track1')

    # modified for outputs
    stats_dict, error_files = run_cycle(args.srclang, args.tgtlang, args.mode, f'../submissions/track1')

    full_score_path = "scores/track1_score_dict_full.json"
    base_score_path = "scores/track1_score_dict.json"

    # If the full file already exists, continue updating it; otherwise start from the base template
    read_path = full_score_path if os.path.exists(full_score_path) else base_score_path

    with open(read_path, "r", encoding="utf-8") as f:
        score_dict = json.load(f)

        mapping_dict = {
            "frequent": "consistency_frequent",
            "predefined": "consistency_predefined"
        }

        for team in stats_dict:
            for pseudoref_choice in mapping_dict:
                # Remove file extension to ensure JSON match correctly
                team_name = team.split("/")[-1].split(".")[0]
                score_dict[args.tgtlang][args.mode][team_name][mapping_dict[pseudoref_choice]] = stats_dict[team][pseudoref_choice]['macro']

        # Save updated scores to the full score file
        with open(full_score_path, "w", encoding="utf-8") as out_f:
            json.dump(score_dict, out_f, indent=4, ensure_ascii=False)

    b = time.time()
    print(f'finished in {b-a} seconds')
    print(error_files)
    with open(f'stats/stats-{args.srclang}{args.tgtlang}.{args.mode}.json', 'w') as fp:
        json.dump(stats_dict, fp)


    # we want [teamname][frequent/predefined][macro]

Overwriting consistency_script_track1.py


In [ ]:
SAVE_PATH = "/content/drive/MyDrive/consistency"

os.makedirs(SAVE_PATH, exist_ok=True)

with open(f"{SAVE_PATH}/track1_score_dict.json", "w") as f:
    # 2. Evaluate consistency
    # Noterm mode
    !python consistency_script_track1.py -s en -t de -m noterm
    # Proper mode
    !python consistency_script_track1.py -s en -t de -m proper
    # Random mode
    !python consistency_script_track1.py -s en -t de -m random

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
en de noterm
['.DS_Store', 'RQ3_SFT', 'RQ4_MultiStep_Base', 'RQ1_Base', 'RQ4_MultiStep_SFT', 'tower']
file_list:  ['RQ3_SFT/RQ3_SFT.ende.noterm.jsonl', 'RQ4_MultiStep_Base/RQ4_MultiStep_Base.ende.noterm.jsonl', 'RQ1_Base/RQ1_Base.ende.noterm.jsonl', 'RQ4_MultiStep_SFT/RQ4_MultiStep_SFT.ende.noterm.jsonl', 'tower/tower.ende.noterm.jsonl']
2026-05-23 10:55:34 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-05-23 10:55:34 INFO: Downloaded file to /root/stanza_resources/resources.json
2026-05-23 10:55:34 WARNING: Language en package default expects mwt, which has been added
2026-05-23 10:55:34 INFO: Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize

Finally, to ensure correct plotting, the following scripts also need to be updated: plot_tradeoff.py and plot_effect_termmode.py.

In [ ]:
%cd /content/nlp2-26/wmt25-terminology/visualization/

In [ ]:
%%writefile plot_tradeoff.py
import json
import matplotlib.pyplot as plt
import os
import statistics

# If utils is missing or throws an error, you can safely comment this out
try:
    import utils
except ImportError:
    utils = None

os.makedirs("generated/", exist_ok=True)
plt.rcParams["font.family"] = "serif"
LANGS = ["de"]

plt.figure(figsize=(5.0, 3.5)) # Slightly adjusted for better label fitting
ax = plt.gca()

# 1. Load the data
with open("ranking/metric_track1/track1_score_dict.json", "r") as f:
    data = json.load(f)

# 2. Extract ChrF++ and Terminology Accuracy (Acc)
plot_data = []
for sys, val in data["de"]["proper"].items():
    if not val:
        continue
    try:
        plot_data.append({
            "name": sys,
            "x": data["de"]["proper"][sys]["proper_term_success_rate"],
            "y": data["de"]["proper"][sys]["chrf2++"]
        })
    except KeyError:
        # Skip if a system is missing one of the metrics
        continue

# 3. Plot the scatter points
plt.scatter(
    [x["x"] for x in plot_data],
    [x["y"] for x in plot_data],
    color='#d37527', # Using a clean highlight color
    marker='o',
    s=100,
    zorder=5,
)

# 4. Add text labels for your specific systems (RQ1, RQ3, RQ4, tower)
for line in plot_data:
    # Use utils mapping if available, otherwise use raw system name
    label_name = line["name"]
    if utils and hasattr(utils, 'SYS_TO_NAME_2'):
        label_name = utils.SYS_TO_NAME_2.get(line["name"], line["name"])

    plt.text(
        line["x"],
        line["y"] + 1.0, # Offset the label slightly above the dot for readability
        label_name,
        fontsize=8,
        ha='center',
        va='bottom',
        zorder=10,
    )

# 5. Formatting
# Set x-axis to show percentages
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x*100)}%'))

# Remove top and right borders for a cleaner academic look
ax.spines[["top", "right"]].set_visible(False)

# Add padding to axes so labels don't get cut off
plt.margins(x=0.15, y=0.15)

plt.xlabel('Terminology Accuracy')
plt.ylabel('ChrF++')

plt.tight_layout(pad=1.0)
plt.savefig("generated/tradeoff_accuracy.pdf")
print("Successfully generated tradeoff_accuracy.pdf!")

In [ ]:
%%writefile plot_effect_termmode.py
import json
import matplotlib.pyplot as plt
import os
import statistics

# Set up parameters
metric_id = "chrf2++"
metric_label = "ChrF++"
limits = (35, 70) # Widened limits slightly to fit your data range (which goes from ~38 to ~65)
file_suffix = ""

os.makedirs("generated/", exist_ok=True)
plt.rcParams["font.family"] = "serif"

plt.figure(figsize=(5.0, 4.0)) # Slightly wider to accommodate labels
ax = plt.gca()

# 1. Load the data
with open("ranking/metric_track1/track1_score_dict.json", "r") as f:
    data = json.load(f)

# The expected structure is: data["de"]["noterm"]["RQ1_Base"]["chrf2++"]
# Get a list of all systems present in the "de" -> "proper" dictionary
all_systems = list(data["de"]["proper"].keys())

plot_data = []

# 2. Extract Data Safely
for sys in all_systems:
    try:
        # Fetch the metric for all three modes. If a mode is missing, this try block catches it.
        val_noterm = data["de"]["noterm"][sys][metric_id]
        val_random = data["de"]["random"][sys][metric_id]
        val_proper = data["de"]["proper"][sys][metric_id]

        # Calculate the mean of these three values to sort them left-to-right on the graph
        mean_val = statistics.mean([val_noterm, val_random, val_proper])

        plot_data.append({
            "name": sys,
            "y": [val_noterm, val_random, val_proper],
            "mean": mean_val
        })
    except KeyError:
        print(f"Skipping {sys} due to missing data for metric {metric_id}")
        continue

# Sort the systems descending by their average score so the best are on the left
plot_data.sort(key=lambda line: line["mean"], reverse=True)

# 3. Plotting
for line_i, line in enumerate(plot_data):
    # Plot 'noterm' (x)
    plt.plot(
        [line_i],
        [line["y"][0]],
        color="black",
        marker="x",
        markersize=6,
        label="No Term" if line_i == 0 else "" # Only add label once for legend
    )

    # Plot 'random' (R) with a white background square for readability
    plt.plot(
        [line_i],
        [line["y"][1]],
        color="white",
        marker="s",
        markersize=8,
        zorder=-5,
    )
    plt.plot(
        [line_i],
        [line["y"][1]],
        color="black",
        marker="$R$",
        markersize=8,
        label="Random" if line_i == 0 else ""
    )

    # Plot 'proper' (Star)
    plt.plot(
        [line_i],
        [line["y"][2]],
        color="black",
        marker=r"$\star$",
        markersize=10,
        label="Proper" if line_i == 0 else ""
    )

    # Draw the vertical line connecting the three points
    plt.plot(
        [line_i]*3,
        line["y"],
        color="black",
        linestyle="--",
        linewidth=1,
        zorder=-10,
    )

# 4. Formatting
plt.ylim(limits)

# Remove top and right spines
ax.spines[["top", "right"]].set_visible(False)

# Set X-axis ticks and labels (rotate 45 degrees so long names like RQ4_MultiStep fit)
plt.xticks(
    range(len(plot_data)),
    [line["name"] for line in plot_data],
    rotation=45,
    ha='right', # Align the end of the text to the tick
    fontsize=9,
)
plt.ylabel(metric_label)

# Add a legend so the reader knows what x, R, and Star mean
plt.legend(loc="lower left", fontsize=8)

# Use pad and tight_layout so nothing gets cut off
plt.tight_layout(pad=1.0)
plt.savefig(f"generated/effect_termmode{file_suffix}.pdf")
print(f"Successfully generated effect_termmode{file_suffix}.pdf!")

In [ ]:
%cd /content/nlp2-26/wmt25-terminology

# 3. Automatically generate tables and figures
# !python visualization/plot_table_track1.py
!python visualization/plot_tradeoff.py
!python visualization/plot_effect_termmode.py

/content/nlp2-26/wmt25-terminology
